# 🤖 Machine Learning Interview Handbook

Этот ноутбук покрывает всю базу классического машинного обучения: от понимания метрик до устройства градиентного бустинга и полного цикла разработки (ML System Design).

---

## Блок 1: Типы задач и Метрики

### Типы задач:
- **Классификация:** Определение дискретного класса (кошка/собака, спам/не спам).
- **Регрессия:** Предсказание непрерывного числа (цена квартиры, температура).
- **Кластеризация:** Разбиение на группы без учителя (сегментация клиентов).
- **Ранжирование:** Сортировка выдачи (поисковые системы, рекомендации).

### 📊 Метрики
**Для классификации:**
- `Accuracy`: Доля правильных ответов (плохо работает при дисбалансе).
- `Precision` (Точность): Из тех, кого алгоритм счет "положительным", сколько реально "положительные"? (Важно, когда ложное срабатывание стоит дорого).
- `Recall` (Полнота): Из всех реальных "положительных", сколько алгоритм нашел? (Важно в медицине — не пропустить болезнь).
- `F1-score`: Гармоническое среднее Precision и Recall, хороший баланс.
- `ROC-AUC`: Площадь под кривой ошибок. Показывает способность модели разделять классы (от 0.5 до 1).

**Для регрессии:**
- `MAE` (Mean Absolute Error): Средняя абсолютная ошибка (устойчива к выбросам).
- `MSE` (Mean Squared Error): Квадратичная ошибка (сильно штрафует за крупные выбросы).
- `RMSE`: Корень из MSE (возвращает ошибку в исходные единицы измерения).
- `R²`: Коэффициент детерминации (какую долю дисперсии объясняет модель; максимум 1).
- `MAPE`: Ошибка в процентах.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = [0, 1, 1, 0, 1, 0]
y_pred = [0, 1, 0, 0, 1, 1]

print(f"Accuracy: {accuracy_score(y_true, y_pred):.2f}")
print(f"Precision: {precision_score(y_true, y_pred):.2f}")
print(f"Recall: {recall_score(y_true, y_pred):.2f}")
print(f"F1: {f1_score(y_true, y_pred):.2f}")

---
## Блок 2: Валидация, Переобучение и Регуляризация

### 🔎 Train / Validation / Test
- `Train`: На этих данных модель учится (подбирает веса).
- `Validation`: На этих настраиваются гиперпараметры и выбирается лучшая модель.
- `Test`: Отложенная выборка для финальной и "честной" проверки (не используется до самого конца).
- **Stratified Split:** Разбиение, при котором сохраняется исходная пропорция классов (жизненно важно при дисбалансе!).

### 🎭 Переобучение (Overfitting) и Недообучение (Underfitting)
- **Overfitting:** Модель "выучила" тренировочные данные наизусть, но плохо обобщает на новые данные (высокая дисперсия, переусложненная модель).
- **Underfitting:** Модель не уловила закономерности даже на трейне (высокое смещение, слишком простая модель).

### 🛡️ Регуляризация (Штраф за сложность)
- **L1 (Lasso):** Штрафует за модули весов. Умеет строго занулять неважные веса (работает как отбор признаков).
- **L2 (Ridge):** Штрафует за квадраты весов. Делает все веса маленькими (защита от мультиколлинеарности).

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

X = np.random.rand(100, 5)
y = np.array([0]*90 + [1]*10) # Имитация сильного дисбаланса (90/10)

# Стратифицированное разбиение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Доля единиц в исходном y: {y.mean():.2f}")
print(f"Доля единиц в тренировочном y: {y_train.mean():.2f}") # Благодаря stratify=y, дисбаланс будет сохранен

---
## Блок 3: Обзор Классических Алгоритмов ML

1. **Линейная / Логистическая регрессия:** 
   - Линейная: предсказывает число ($y = wx + b$). 
   - Логистическая: для классификации. Пропускает выход через сигмоиду для получения вероятности от 0 до 1. **Требует масштабирования признаков (Scaler)**.
2. **KNN (K-Nearest Neighbors):** Алгоритм без обучения. Ищет K ближайших точек. Зависит от метрики расстояния. Боится высокой размерности ("проклятие размерности"). Требует стандартизации.
3. **SVM (Support Vector Machine):** Ищет гиперплоскость, максимизирующую зазор (margin) между классами. Использует ядро (kernel trick) для перехода в новые пространства (для линейно неразделимых данных).
4. **Деревья решений:** Алгоритм, делящий пространство признаками. Легко переобучается, если не ограничить глубину.
5. **Ансамбли деревьев (RandomForest & Gradient Boosting):**
   - **RandomForest (Бэггинг):** Деревья строятся независимо друг от друга на случайных подвыборках (bootstrap) и случайных признаках. Итог - голосование. Уменьшает дисперсию (борется с переобучением базовых деревьев).
   - **Gradient Boosting (Бустинг):** Деревья строятся последовательно. Каждое следующее исправляет ошибки предыдущего (используя градиент функции потерь). Очень мощный алгоритм, но склонен к переобучению, если не настроить learning_rate.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Пример настройки случайного леса
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# Ансамбли на основе деревьев хороши тем, что показывают важность признаков
print("Важность признаков:", rf.feature_importances_)

---
## Блок 4: Подбор гиперпараметров и Сохранение модели

Выбор оптимальных параметров — ключи к хорошей модели.

- **GridSearchCV:** Полный перебор всех возможных комбинаций по заданной сетке. Идеально находит лучший вариант, но очень долго работает.
- **RandomizedSearchCV:** Случайный перебор заданного числа комбинаций. Работает в разы быстрее, а результат часто сравним с GridSearch.
- **Сохранение:** Обученную модель необходимо сохранять (сериализация) с помощью библотек `Pickle` или `Joblib`.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
import joblib

# Поиск оптимального гиперпараметра C (сила регуляризации)
params = {'C': [0.01, 0.1, 1.0, 10.0, 100.0]}
search = RandomizedSearchCV(LogisticRegression(), params, n_iter=3, cv=3, random_state=42)
search.fit(X_train, y_train)

print("Лучший найденный параметр C:", search.best_params_['C'])

# --- Сохранение модели на диск --- 
best_model = search.best_estimator_
joblib.dump(best_model, 'best_logreg_model.pkl')
print("Модель сохранена как best_logreg_model.pkl")

# Как потом загрузить (в продакшене):
# loaded_model = joblib.load('best_logreg_model.pkl')

---
## Блок 5: Полный цикл ML задачи (ML System Design Lifecycle)

Классический вопрос на System Design интервью — "Опишите процесс решения ML-задачи от начала до конца":

1. **Бизнес-понимание:** Какую проблему мы решаем? Что будет метрикой успеха для бизнеса?
2. **Сбор и разметка данных (Data Collection):** Откуда берем данные? Хватает ли их качества? Как будем размечать (Толока, эксперты)?
3. **EDA и Препроцессинг:** Очистка от пропусков, удаление выбросов, лог-преобразования. 
4. **Генерация фичей (Feature Engineering):** Создание новых полезных колонок.
5. **Выбор модели и Обучение:** Начинаем с простых бейзлайнов (LogReg). Если бейзлайн работает - пробуем усложнять (CatBoost, LightGBM).
6. **Валидация (Evaluation):** Строгая проверка на `Validation` и `Test` сплитах. Анализ ошибок модели (почему она ошибается на конкретных примерах?).
7. **Деплой (Deployment):** Оборачивание модели в REST API микросервис (например, FastAPI, Flask + Docker).
8. **Мониторинг (Monitoring):** Модели со временем теряют актуальность (Data Drift / Concept Drift). Нужно мониторить "здоровье" модели и периодически ее дообучать.

### 📝 Контрольные вопросы для самопроверки перед интервью

1. Почему `Accuracy` нельзя использовать, если в выборке 99% нормальных транзакций и 1% мошеннических?
2. В чем фундаментальное отличие процесса построения деревьев в Random Forest и Gradient Boosting?
3. Как именно L1 регуляризация помогает отобрать только важные признаки?
4. Зачем в KNN обязательно выполнять масштабирование (StandardScaler) признаков перед обучением?

---
## 🎯 Реальные вопросы с собеседований (Middle Data Scientist)

### 🎤 Вопросы по моделям:
1. **Трансформация признаков:** Вы умножили все значения непрерывного признака (например, рост в см) на 100. Изменится ли от этого предсказание: а) Линейной регрессии? б) Дерева Решений? (Ответ обоснуйте математически).
2. **Смещение и дисперсия (Bias-Variance Tradeoff):** Если модель состоит из очень глубоких деревьев решений (без ограничения глубины), у нее высокое смещение или высокая дисперсия? Что произойдет, если мы объединим их в Random Forest?
3. **Предпосылки Логистической Регрессии:** Какие предположения о данных делает логистическая регрессия? (Ожидаемый ответ: отсутствие строгой мультиколлинеарности, линейная разделимость в пространстве признаков или трансформаций).
4. **Дисбаланс классов:** Метрика ROC-AUC показала 0.95 на тестовой выборке (классы 99 к 1). Можете ли вы сказать, что модель отлично ловит миноритарный класс? Почему здесь лучше смотреть на Precision-Recall AUC?

### 💻 Архитектурная задача (System Design ML):
Опишите пайплайн разработки рекомендательной системы фильмов. Какую архитектуру (двухэтапную: Candidate Retrieval + Ranking) вы бы использовали и какие метрики в оффлайне (NDCG) и в онлайне (A/B тест на кликабельность) вы бы выбрали?

### ✅ Ответы на классические вопросы ML

<details>
<summary><b>Ответ 1: Bias-Variance Tradeoff</b></summary>

- **Bias (Смещение):** Ошибка из-за того, что модель слишком простая (Underfitting) и не может описать данные.
- **Variance (Дисперсия):** Ошибка из-за того, что модель слишком сложная (Overfitting) и заучивает шум/выбросы.
- **Трейд-офф:** Идеальная модель находится на балансе. При усложнении Bias падает, но Variance растет (и наоборот).
</details>

<details>
<summary><b>Ответ 2: Random Forest vs Gradient Boosting</b></summary>

Оба ансамбли из деревьев, но:
- **Random Forest (Bagging):** Деревья строятся **параллельно** и независимо на случайных подвыборках данных. Хорошо борется с Variance (переобучением).
- **Gradient Boosting (Boosting):** Деревья строятся **последовательно**. Каждое следующее дерево исправляет ошибки предыдущего. Хорошо борется с Bias (недообучением). Более мощный, но склонен к переобучению.
</details>

<details>
<summary><b>Ответ 3: Метрики при дисбалансе</b></summary>

При дисбалансе классов (например, 99% здоровых, 1% больных) модель может предсказывать всем "Здоров" и получать Accuracy 99%.
В таких случаях смотрят на **Precision** (Точность, чтобы не ругаться впустую), **Recall** (Полнота, чтобы не пропустить больного), и объединяющую их метрику **F1-Score**. Для оценки алгоритма в целом — площадь под **PR Curve** или **ROC-AUC**.
</details>

---
## 🧠 Частые дополнительные вопросы (на понимание)

<details>
<summary><b>1. Разница между L1 (Lasso) и L2 (Ridge) регуляризацией</b></summary>

Обе штрафуют модель за слишком большие веса, чтобы избежать переобучения.
- **L1 (Lasso):** Использует модуль весов. Обладает свойством **отбора признаков** (зануляет веса неинформативных признаков).
- **L2 (Ridge):** Использует квадрат весов. Приближает веса к нулю, но редко зануляет полностью. Хорошо работает при сильной корреляции признаков.
</details>

<details>
<summary><b>2. Стратегии кросс-валидации</b></summary>

- **K-Fold:** Данные разбиваются на $K$ частей, модель обучается $K$ раз. Оценка надежна.
- **Stratified K-Fold:** Важен для задач классификации с дисбалансом. Камера следит за тем, чтобы соотношение классов во всех фолдах было одинаковым.
- **Time Series Split:** Использовать обычный K-fold для временных рядов нельзя (нельзя тестировать модель "в прошлом", обучив на "будущем"). Разделение делается так, чтобы обучаться на более старых данных, а тестироваться на более новых.
</details>